In [4]:
"""Cover fractions and 20% labels for every 1000, 500 and 250 px tile.

Reads the "tiles" and "rhodo" layers from labelling.gpkg and writes
tiles_1000px.gpkg, tiles_500px.gpkg and tiles_250px.gpkg.

Usage: python cover_fractions.py labelling.gpkg
Close QGIS (or save all edits) before running.
"""
import sys

import geopandas as gpd
from shapely.geometry import box

LABELLING_PATH = "../Grids/labelling.gpkg"
COVER_THRESHOLD = 0.20                   # cover >= 20% -> positive
LABELLED_STATUSES = ["done", "unsure"]   # "skip" and unlabelled tiles are left out
SIZES = [(1000, 1), (500, 2), (250, 4)]  # (tile size in px, splits per side)

tiles = gpd.read_file(LABELLING_PATH, layer="tiles")
rhododendron_outlines = gpd.read_file(LABELLING_PATH, layer="rhododendron_outlines")
rhododendron_outlines["geometry"] = rhododendron_outlines.make_valid()
rhodo_area = rhododendron_outlines.union_all()           # merge overlaps so no area counts twice

labelled = tiles[tiles["status"].isin(LABELLED_STATUSES)]
print(f"{len(labelled)} labelled tiles, {(tiles['status'] == 'skip').sum()} skipped, "
      f"{tiles['status'].isna().sum()} with no status")

keep = [c for c in tiles.columns if c != "geometry"]
rows = {size: [] for size, _ in SIZES}
for tile in labelled.itertuples(index=False):
    min_x, min_y, max_x, max_y = tile.geometry.bounds
    rhodo_in_tile = rhodo_area.intersection(tile.geometry)   # clip once per tile
    for size, splits in SIZES:
        step_x, step_y = (max_x - min_x) / splits, (max_y - min_y) / splits
        for sub_row in range(splits):                         # row 0 = top
            for sub_col in range(splits):
                sub = box(min_x + sub_col * step_x, max_y - (sub_row + 1) * step_y,
                          min_x + (sub_col + 1) * step_x, max_y - sub_row * step_y)
                cover = rhodo_in_tile.intersection(sub).area / sub.area
                label = int(cover >= COVER_THRESHOLD)
                rows[size].append({**{c: getattr(tile, c) for c in keep},
                                   "tile_size_px": size, "sub_row": sub_row, "sub_col": sub_col,
                                   "cover_fraction": round(cover, 4), "label": label,
                                   "class": "positive" if label else "negative",
                                   "geometry": sub})

for size, _ in SIZES:
    out = gpd.GeoDataFrame(rows[size], geometry="geometry", crs=tiles.crs)
    path = f"../Grids/tiles_{size}px.gpkg"
    out.to_file(path, layer=f"tiles_{size}px", driver="GPKG")
    print(f"{path}: {len(out)} tiles, {out['label'].sum()} positive "
          f"({out['label'].mean():.1%}), mean cover {out['cover_fraction'].mean():.3f}")

500 labelled tiles, 0 skipped, 4681 with no status
../Grids/tiles_1000px.gpkg: 500 tiles, 49 positive (9.8%), mean cover 0.059
../Grids/tiles_500px.gpkg: 2000 tiles, 174 positive (8.7%), mean cover 0.059
../Grids/tiles_250px.gpkg: 8000 tiles, 604 positive (7.5%), mean cover 0.059


In [5]:
import geopandas as gpd
from sklearn.model_selection import StratifiedGroupKFold

FOLD_SEED = 20261114   # record this in your methods
N_FOLDS = 5
SIZES = [1000, 500, 250]

tiles = {s: gpd.read_file(f"../Grids/tiles_{s}px.gpkg") for s in SIZES}

# Assign folds by block, using the 1000 px tiles (KD02, training role, "done" only)
parents = tiles[1000]
parents = parents[(parents["forest"] == "KD02") & (parents["role"] == "train") & (parents["status"] == "done")]
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=FOLD_SEED)
fold_of_block = {}
for fold, (_, test_idx) in enumerate(cv.split(parents, parents["label"], groups=parents["block"])):
    for block in parents.iloc[test_idx]["block"].unique():
        fold_of_block[block] = fold

# Same block -> same fold at every size; save it into each GeoPackage
for s in SIZES:
    tiles[s]["fold"] = tiles[s]["block"].map(fold_of_block)   # NaN = not in stage A
    tiles[s].to_file(f"../Grids/tiles_{s}px.gpkg", layer=f"tiles_{s}px", driver="GPKG")
    used = tiles[s][tiles[s]["fold"].notna() & (tiles[s]["status"] == "done")]
    print(f"\n{s} px")
    print(used.groupby("fold").agg(blocks=("block", "nunique"), tiles=("label", "size"),
                                   positives=("label", "sum"), share_pos=("label", "mean")).round(3))


1000 px
      blocks  tiles  positives  share_pos
fold                                     
0          3     76          7      0.092
1          3     69          5      0.072
2          4     97          8      0.082
3          3     79          8      0.101
4          4     99         20      0.202

500 px
      blocks  tiles  positives  share_pos
fold                                     
0          3    304         27      0.089
1          3    276         15      0.054
2          4    388         27      0.070
3          3    316         25      0.079
4          4    396         76      0.192

250 px
      blocks  tiles  positives  share_pos
fold                                     
0          3   1216         97      0.080
1          3   1104         51      0.046
2          4   1552         85      0.055
3          3   1264         90      0.071
4          4   1584        262      0.165


In [6]:
# Positives per block at 1000 px
used = parents.assign(fold=parents["block"].map(fold_of_block))
print(used.groupby(["fold", "block"])["label"].agg(tiles="size", positives="sum").to_string())

# Try a range of seeds and keep the one with the most even positives across folds
import numpy as np
results = []
for seed in range(100):
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    pos = [parents.iloc[te]["label"].sum() for _, te in cv.split(parents, parents["label"], groups=parents["block"])]
    results.append((seed, min(pos), max(pos), np.std(pos)))
best = sorted(results, key=lambda r: r[3])[:5]
print("seed, min positives, max positives, std:", best)

                tiles  positives
fold block                      
0    KD02_2001     35          1
     KD02_2003     32          6
     KD02_2005      9          0
1    KD02_2000     30          0
     KD02_3002     30          5
     KD02_4001      9          0
2    KD02_0004     17          0
     KD02_1001     22          0
     KD02_2004     30          8
     KD02_4003     28          0
3    KD02_1002     23          5
     KD02_1004     28          0
     KD02_3003     28          3
4    KD02_0003     18          0
     KD02_1003     31         17
     KD02_3001     33          3
     KD02_3004     17          0
seed, min positives, max positives, std: [(0, np.int64(6), np.int64(20), np.float64(5.276362383309167)), (1, np.int64(6), np.int64(20), np.float64(5.276362383309167)), (6, np.int64(6), np.int64(20), np.float64(5.276362383309167)), (7, np.int64(6), np.int64(20), np.float64(5.276362383309167)), (10, np.int64(6), np.int64(20), np.float64(5.276362383309167))]


In [7]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

TIFF_PATH = "../LargeImages/KD02-1-20241203-7_IXM150_mos.tif"
GRID_DIR = Path("../Grids")
OUT_DIR = Path("../ImageFiles/InitialComparison")
SPLITS = {1000: 1, 500: 2, 250: 4}       # sub-tiles per side

# Stage A tiles: KD02 training blocks, "done" only, with a fold
parents = gpd.read_file(GRID_DIR / "tiles_1000px.gpkg")
parents = parents[(parents["forest"] == "KD02") & (parents["role"] == "train")
                  & (parents["status"] == "done") & parents["fold"].notna()]
print(f"{len(parents)} tiles -> {len(parents) * 21} images")

written, problems = 0, []
with rasterio.open(TIFF_PATH) as ds:
    for tile in tqdm(parents.itertuples(), total=len(parents)):
        window = from_bounds(*tile.geometry.bounds, transform=ds.transform).round_offsets().round_lengths()
        rgb = ds.read([1, 2, 3], window=window, boundless=True, fill_value=0)   # bands 1-3 only
        rgb = np.moveaxis(rgb, 0, -1)                                            # (rows, cols, 3)
        if rgb.shape[:2] != (1000, 1000):
            problems.append((tile.tile_id, rgb.shape))
            continue
        for size, n in SPLITS.items():
            for sub_row in range(n):                                 # row 0 = top, as in the GeoPackages
                for sub_col in range(n):
                    img = rgb[sub_row * size:(sub_row + 1) * size, sub_col * size:(sub_col + 1) * size]
                    name = f"{tile.tile_id}_{size}_{sub_row}_{sub_col}.png"
                    Image.fromarray(img).save(OUT_DIR / str(size) / name)
                    written += 1

print(f"Saved {written} images")
if problems:
    print(f"{len(problems)} tiles not 1000 x 1000 px, skipped:", problems[:10])

/home/cyan/miniconda3/envs/RhodaiResit/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


420 tiles -> 8820 images


100%|██████████| 420/420 [05:38<00:00,  1.24it/s]

Saved 8820 images


In [2]:
import tensorflow as tf; print(tf.__version__)

2.16.2


In [4]:
from train import CONFIG, run
cfg = dict(CONFIG, tile_size_px=1000, seeds=[0], folds=[0],
           batch_size=8,            # 1000 px images are large; drop to 4 if you run out of memory
           epochs=40,               # longer than you'll need, to see where the loss flattens
           run_name="test_1000")    # so the real run's file name stays free
run(cfg)

Run test_1000  started 2026-11-14 16:23
Settings: {"run_name": "test_1000", "tile_size_px": 1000, "max_order": null, "seeds": [0], "folds": [0], "grid_dir": "../Grids", "image_dir": "../ImageFiles/InitialComparison", "results_dir": "../Results", "forest": "KD02", "statuses": ["done"], "greyscale": false, "resize_to_px": null, "weights": "imagenet", "train_base": true, "dropout": 0.2, "learning_rate": 0.0001, "batch_size": 8, "epochs": 40, "steps_per_epoch": 50, "class_weights": true, "augment": true, "contrast_range": [0.8, 1.2], "brightness_delta": 0.1, "threshold": 0.5, "save_models": false, "save_predictions": false}
420 images, 48 positive, folds [0]

--- seed 0, fold 0: train 344 (41 pos), test 76 (7 pos), class weights {0: 0.568, 1: 4.195}


/home/cyan/Projects/RhodaiResit/Scripts/train.py:125: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = keras.applications.MobileNetV2(input_shape=(size, size, 3), include_top=False,
2026-11-14 16:23:47.446656: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-11-14 16:23:47.745266: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-11-14 16:23:47.7

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step 
Epoch 1/40


/home/cyan/miniconda3/envs/RhodaiTF/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-11-14 16:24:08.268195: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 134217984 bytes after encountering the first element of size 134217984 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
2026-11-14 16:24:08.467307: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 134218016 bytes after encountering the first element of size 134218016 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increas

KeyboardInterrupt: 